In [1]:
# Load environment variables from project root
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [2]:
# Create Databricks Spark session
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.silver')

DataFrame[]

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType, ArrayType

# Read bronze source tables
bronze_questions = spark.table('ddc_databricks.bronze.so_questions')
bronze_answers = spark.table('ddc_databricks.bronze.so_answers')
bronze_tag_info = spark.table('ddc_databricks.bronze.so_tag_info')

In [5]:
# 1) Questions -> silver.so_questions
html_code_pattern = F.lit(r'(?is)<pre><code>.*?</code></pre>')
html_link_pattern = F.lit(r'(?is)<a\s+[^>]*href=')

questions_flat = (
    bronze_questions
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.col('_so_tag').alias('so_tag'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.explode_outer('questions').alias('q')
    )
    .withColumn('q_json', F.to_json(F.col('q')))
    .select(
        'package_name',
        'so_tag',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.col('q.question_id').cast('long').alias('question_id'),
        F.from_unixtime(F.col('q.creation_date')).cast('timestamp').alias('question_created_at'),
        F.from_unixtime(F.col('q.last_activity_date')).cast('timestamp').alias('question_last_activity_at'),
        F.from_unixtime(F.col('q.last_edit_date')).cast('timestamp').alias('question_last_edit_at'),
        F.from_unixtime(F.col('q.closed_date')).cast('timestamp').alias('question_closed_at'),
        F.col('q.score').cast('long').alias('question_score'),
        F.col('q.view_count').cast('long').alias('question_view_count'),
        F.col('q.answer_count').cast('long').alias('question_answer_count'),
        F.get_json_object(F.col('q_json'), '$.favorite_count').cast('long').alias('question_favorite_count'),
        F.col('q.is_answered').cast('boolean').alias('is_answered'),
        F.col('q.tags').alias('question_tags'),
        F.col('q.title').alias('question_title'),
        F.col('q.body').alias('question_body_html'),
        F.col('q.owner.user_id').cast('long').alias('owner_user_id'),
        F.col('q.owner.display_name').alias('owner_display_name'),
        F.col('q.owner.reputation').cast('long').alias('owner_reputation')
    )
    .filter(F.col('question_id').isNotNull())
    .withColumn('question_title_char_count', F.length(F.coalesce(F.col('question_title'), F.lit(''))))
    .withColumn('question_body_char_count', F.length(F.coalesce(F.col('question_body_html'), F.lit(''))))
    .withColumn('question_body_code_block_count', F.regexp_count(F.col('question_body_html'), html_code_pattern))
    .withColumn('question_body_link_count', F.regexp_count(F.col('question_body_html'), html_link_pattern))
    .withColumn('question_tags_count', F.size(F.coalesce(F.col('question_tags'), F.array())))
)

questions_dedup_w = Window.partitionBy('package_name', 'question_id').orderBy(F.col('ingested_at').desc(), F.col('question_last_activity_at').desc())
questions_latest = (
    questions_flat
    .withColumn('rn', F.row_number().over(questions_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    questions_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_questions')
)

print('silver.so_questions created')

silver.so_questions created


In [10]:
# 2) Answers -> silver.so_answers
html_code_pattern = F.lit(r'(?is)<pre><code>.*?</code></pre>')
html_link_pattern = F.lit(r'(?is)<a\s+[^>]*href=')

answers_flat = (
    bronze_answers
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.col('_so_tag').alias('so_tag'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.explode_outer('answers').alias('a')
    )
    .select(
        'package_name',
        'so_tag',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.col('a.answer_id').cast('long').alias('answer_id'),
        F.col('a.question_id').cast('long').alias('question_id'),
        F.from_unixtime(F.col('a.creation_date')).cast('timestamp').alias('answer_created_at'),
        F.from_unixtime(F.col('a.last_activity_date')).cast('timestamp').alias('answer_last_activity_at'),
        F.from_unixtime(F.col('a.last_edit_date')).cast('timestamp').alias('answer_last_edit_at'),
        F.col('a.score').cast('long').alias('answer_score'),
        F.col('a.is_accepted').cast('boolean').alias('is_accepted'),
        F.col('a.body').alias('answer_body_html'),
        F.col('a.owner.user_id').cast('long').alias('owner_user_id'),
        F.col('a.owner.display_name').alias('owner_display_name'),
        F.col('a.owner.reputation').cast('long').alias('owner_reputation')
    )
    .filter(F.col('answer_id').isNotNull())
    .withColumn('answer_body_char_count', F.length(F.coalesce(F.col('answer_body_html'), F.lit(''))))
    .withColumn('answer_body_code_block_count', F.regexp_count(F.col('answer_body_html'), html_code_pattern))
    .withColumn('answer_body_link_count', F.regexp_count(F.col('answer_body_html'), html_link_pattern))
)

answers_dedup_w = Window.partitionBy('package_name', 'answer_id').orderBy(F.col('ingested_at').desc(), F.col('answer_last_activity_at').desc())
answers_latest = (
    answers_flat
    .withColumn('rn', F.row_number().over(answers_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    answers_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_answers')
)

print('silver.so_answers created')

silver.so_answers created
